# Cats vs. Dogs — Train & Export VGG16 on Kaggle 🐾

A **Kaggle-ready** version of the working VGG16 transfer-learning experiment from `transfer-learning_true.ipynb`.

### Before running
1. Create a Kaggle notebook, then **File → Import Notebook** and upload this `.ipynb`.
2. Under **Settings / Session options**, select **Accelerator → GPU T4 x2**.
3. Switch **Internet → On** (needed for TensorFlow Datasets and ImageNet VGG16 weights). Kaggle may require account/phone verification to enable this.
4. **Run All** from the beginning; don't run just the export cell.

**What you'll get:** `/kaggle/working/cats_dogs_vgg16.keras` (trained model) and `/kaggle/working/test_metrics.json` (new evaluation results). Download these from Kaggle **Output / Files** before leaving the runtime, or save a notebook version with outputs.

The 97.56% held-out accuracy observed in your **original** notebook is not guaranteed for a fresh training run. This notebook evaluates **your newly trained** model.

### TFDS troubleshooting
If you see `AttributeError: module 'tensorflow_datasets' has no attribute 'load'`, the preflight cell will attempt to repair the package and **intentionally stop**. Use Kaggle's **Restart Session/Kernel**, then **Run All**. If the module path still looks wrong, check for a local `tensorflow_datasets.py` file or `tensorflow_datasets/` directory shadowing the installed package.


In [ ]:
# Confirm TensorFlow Datasets actually works (not just that the module exists).
# The earlier version of this notebook checked find_spec(), which can return
# a module even if tfds.load is missing.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])

try:
    import tensorflow_datasets as tfds
    tfds_healthy = callable(getattr(tfds, "load", None))
    print("TFDS module location:", getattr(tfds, "__file__", None))
    print("TFDS version:", getattr(tfds, "__version__", "unknown"))
    print("TFDS load available:", tfds_healthy)
except Exception as exc:
    tfds_healthy = False
    print("TFDS import failed:", repr(exc))

if not tfds_healthy:
    print("Repairing TFDS without replacing Kaggle's TensorFlow dependencies...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "--force-reinstall", "--no-deps", "tensorflow-datasets==4.9.10"
    ])
    raise RuntimeError(
        "TFDS was repaired. RESTART the Kaggle session/kernel now, then Run All. "
        "If tfds.load is still missing afterward, check whether a local "
        "tensorflow_datasets.py or tensorflow_datasets/ folder shadows the package."
    )


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path
import json

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU detected. In Kaggle, open Settings → Accelerator → GPU T4 x2, then reconnect and rerun.")

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Step 1 — Download the Cats vs. Dogs dataset and reproduce the 3,000-image split

The dataset is fetched through TensorFlow Datasets, just like your original notebook. **Internet must be enabled**. This downloads the dataset **inside Kaggle**, not to your PC. The 70/15/15 train/validation/test split uses the same seed and class order.


In [ ]:
raw_ds, ds_info = tfds.load("cats_vs_dogs", split="train", with_info=True, as_supervised=True)
print("Dataset classes:", ds_info.features["label"].names)  # ['cat', 'dog']

N_PER_CLASS = 1500
raw_balanced_ds = raw_ds.filter(lambda image, label: label == 0).take(N_PER_CLASS).concatenate(
    raw_ds.filter(lambda image, label: label == 1).take(N_PER_CLASS)
)
TOTAL_IMAGES = N_PER_CLASS * 2
N_TRAIN = 2100
N_VAL = 450
N_TEST = 450
shuffled_ds = raw_balanced_ds.shuffle(
    buffer_size=TOTAL_IMAGES, seed=SEED, reshuffle_each_iteration=False
)
train_raw_ds = shuffled_ds.take(N_TRAIN)
val_raw_ds = shuffled_ds.skip(N_TRAIN).take(N_VAL)
test_raw_ds = shuffled_ds.skip(N_TRAIN + N_VAL).take(N_TEST)
print("train / validation / test:", N_TRAIN, N_VAL, N_TEST)

## Step 2 — Exactly match the original VGG16 preprocessing

Use `tf.image.resize` (224 × 224) followed by `keras.applications.vgg16.preprocess_input`. **Do not divide pixels by 255.** The deployed Streamlit app uses the same transformations.


In [ ]:
IMG_SIZE_VGG = (224, 224)
BATCH_SIZE = 32

def vgg_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_VGG)
    image = keras.applications.vgg16.preprocess_input(image)
    return image, label

vgg_train_ds = (
    train_raw_ds.map(vgg_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors().shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)
vgg_val_ds = (
    val_raw_ds.map(vgg_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)
vgg_test_ds = (
    test_raw_ds.map(vgg_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)

## Step 3 — Train the original architecture

ImageNet-pretrained **VGG16 (frozen)** → Flatten → Dropout(0.3) → sigmoid. Adam 1e-3; 8 epochs; batch size 32. Kaggle T4 x2 offers two GPUs, but this standard Keras code uses a single GPU; it doesn't automatically combine both.


In [ ]:
vgg_tl_base = keras.applications.VGG16(
    include_top=False, weights="imagenet", input_shape=(224, 224, 3)
)
vgg_tl_base.trainable = False
vgg_tl_model = keras.Sequential([
    vgg_tl_base,
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="vgg_transfer")
vgg_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
vgg_tl_model.summary()

In [ ]:
history = vgg_tl_model.fit(
    vgg_train_ds, validation_data=vgg_val_ds, epochs=8, verbose=2
)

## Step 4 — Evaluate the newly trained model

Keep the reported test metrics from this run, not the previous notebook's number. A test split is held out until now.


In [ ]:
test_loss, test_acc = vgg_tl_model.evaluate(vgg_test_ds, verbose=0)
y_test = np.concatenate([y for _, y in tfds.as_numpy(vgg_test_ds)])
y_prob = vgg_tl_model.predict(vgg_test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
report = classification_report(y_test, y_pred, target_names=["cat", "dog"], digits=3, output_dict=True)
cm = confusion_matrix(y_test, y_pred)
print(f"Test loss: {test_loss:.4f}, test accuracy: {test_acc:.2%}")
print(classification_report(y_test, y_pred, target_names=["cat", "dog"], digits=3))
print("Confusion matrix:\n", cm)

metrics_path = OUTPUT_DIR / "test_metrics.json"
metrics = {
    "model": "VGG16 frozen-backbone transfer learning",
    "dataset": "TFDS cats_vs_dogs (1,500 images per class sampled)",
    "expected_split": {"train": 2100, "validation": 450, "test": 450},
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "labels": ["cat", "dog"],
    "preprocessing": "tf.image.resize(224,224) + keras.applications.vgg16.preprocess_input"
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved evaluation metrics to:", metrics_path)


## Step 5 — Save and verify the trained model

Save **after training**, never after accidentally rebuilding the classifier. This file includes trained weights and is the file the Streamlit app loads.


In [ ]:
model_file = OUTPUT_DIR / "cats_dogs_vgg16.keras"
vgg_tl_model.save(str(model_file))
reloaded_model = keras.models.load_model(str(model_file), compile=False)

sample_x, _ = next(iter(vgg_test_ds))
pred_original = vgg_tl_model(sample_x, training=False).numpy()
pred_reloaded = reloaded_model(sample_x, training=False).numpy()
np.testing.assert_allclose(pred_original, pred_reloaded, rtol=1e-5, atol=1e-6)
print(f"✅ Saved and verified: {model_file} ({model_file.stat().st_size / 1024**2:.1f} MiB)")


## Step 6 — Download your model from Kaggle

After the last cell finishes, **click the file links below**, or open Kaggle's **Output / Files** pane and download `cats_dogs_vgg16.keras` and `test_metrics.json` from `/kaggle/working/`.

You can also use **Save Version** to persist notebook outputs. Don't rely on a live Kaggle session to keep files forever.

Put the `.keras` file in `cats-dogs-live-demo/models/` on your PC, then run the Streamlit app. Keep `test_metrics.json` for your portfolio's evaluation evidence.


In [ ]:
from IPython.display import display, FileLink

for file_path in [model_file, metrics_path]:
    if file_path.is_file():
        display(FileLink(str(file_path), result_html_prefix=f"Download {file_path.name}: "))
    else:
        print("Missing:", file_path, "— run the prior cells in order.")

print("Kaggle Output folder:", OUTPUT_DIR)
